In [9]:
import struct
import pyodbc
import pandas as pd
from azure.identity import AzureCliCredential

credential = AzureCliCredential()
token = credential.get_token("https://database.windows.net/.default")
token_bytes = token.token.encode("utf-16-le")
token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)

SQL_COPT_SS_ACCESS_TOKEN = 1256

conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=tcp:sqlserver-diabetes-fatima.database.windows.net,1433;"
    "Database=sqldb-diabetes-silver;"
)

conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct}, timeout=30)

df = pd.read_sql("""
    SELECT p.patient_nbr, p.race, p.gender, p.age, e.time_in_hospital, e.num_medications, r.readmitted 
    FROM dbo.patients p
    JOIN dbo.encounters e ON p.patient_nbr = e.patient_nbr
    JOIN dbo.readmission_labels r ON e.encounter_id = r.encounter_id
""", conn)

print(df.shape)
print(df.head(10))

Error: ('HY000', "[HY000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Database 'sqldb-diabetes-silver' on server 'sqlserver-diabetes-fatima.database.windows.net' is not currently available.  Please retry the connection later.  If the problem persists, contact customer support, and provide them the session tracing ID of '{27C43F7F-9CF5-4B49-BE40-F802575AC444}'. (40613) (SQLDriverConnect)")

In [10]:
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from azure.ai.ml import MLClient

# 1. Connect to Azure ML Workspace to get the MLflow tracking URI
ml_client = MLClient(
    credential=credential,  # reuse the AzureCliCredential from the cell above
    subscription_id="dcb792ae-5b43-4cfd-b612-b35b0dad827f",
    resource_group_name="rg-readmission-project",
    workspace_name="aml-diabetes-fatima"
)

mlflow.set_tracking_uri(ml_client.workspaces.get(name="aml-diabetes-fatima").mlflow_tracking_uri)
mlflow.set_experiment("diabetes_readmission_eda")

# 2. Clean up the readmitted column (defensive, in case of whitespace)
df['readmitted'] = df['readmitted'].str.strip()

with mlflow.start_run(run_name="real_eda_run_v2"):
    
    # Log basic parameters
    mlflow.log_param("total_records", len(df))
    mlflow.log_param("missing_race_count", int(df['race'].isna().sum()))
    
    # Log class counts as metrics
    class_counts = df['readmitted'].value_counts()
    print("Class distribution:")
    print(class_counts)
    for item_class, count in class_counts.items():
        mlflow.log_metric(f"class_count_{item_class}", float(count))
    
    # Plot 1: Class imbalance
    plt.figure(figsize=(7, 5))
    sns.countplot(x='readmitted', data=df, order=['NO', '>30', '<30'], palette='viridis')
    plt.title('Patient Readmission Distribution (Target Class)')
    plt.xlabel('Readmitted State')
    plt.ylabel('Patient Record Volume')
    plt.tight_layout()
    plt.savefig("class_imbalance.png")
    mlflow.log_artifact("class_imbalance.png")
    plt.show()  # <-- this displays it right in the notebook so you can see it immediately
    plt.close()
    
    # Plot 2: Medications vs readmission
    plt.figure(figsize=(9, 5))
    sns.boxplot(x='readmitted', y='num_medications', data=df, order=['NO', '>30', '<30'], palette='muted')
    plt.title('Medications Prescribed by Readmission Status')
    plt.tight_layout()
    plt.savefig("medication_impact.png")
    mlflow.log_artifact("medication_impact.png")
    plt.show()
    plt.close()
    
    print("EDA complete — metrics and charts logged to MLflow.")

NameError: name 'df' is not defined

In [7]:
import os

os.makedirs("./data_snapshot", exist_ok=True)
df.to_parquet("./data_snapshot/diabetes_readmission_features.parquet", index=False)
print("Saved locally.")

NameError: name 'df' is not defined

In [4]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_asset = Data(
    name="diabetes_readmission_features",
    version="1",
    description="Joined patients + encounters + readmission_labels, post-normalization fix",
    type=AssetTypes.URI_FILE,
    path="./data_snapshot/diabetes_readmission_features.parquet"
)

ml_client.data.create_or_update(data_asset)
print("Data asset registered.")

Uploading diabetes_readmission_features.parquet (< 1 MB): 100%|██████████| 882k/882k [00:00<00:00, 15.8MB/s]




Data asset registered.


In [5]:
retrieved = ml_client.data.get(name="diabetes_readmission_features", version="1")
print(retrieved.path)

azureml://subscriptions/dcb792ae-5b43-4cfd-b612-b35b0dad827f/resourcegroups/rg-readmission-project/workspaces/aml-diabetes-fatima/datastores/workspaceblobstore/paths/LocalUpload/46862bc720a3384753d337a8bc92d79aed479eaf2581b41b0770ecf4e7335bb7/diabetes_readmission_features.parquet


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load from the registered Data Asset (not a fresh SQL query — this is the point of registering it)
data_asset = ml_client.data.get(name="diabetes_readmission_features", version="1")
df = pd.read_parquet(data_asset.path)

print("Loaded:", df.shape)

# --- Clean/encode ---

# 1. Age: convert bracket strings like "[80-90)" into a numeric midpoint (ordinal-ish, models handle this better than raw text)
def age_to_midpoint(age_bracket):
    low, high = age_bracket.strip("[)").split("-")
    return (int(low) + int(high)) / 2

df["age_numeric"] = df["age"].apply(age_to_midpoint)

# 2. Race/gender: one-hot encode (turns each category into its own 0/1 column)
df = pd.get_dummies(df, columns=["race", "gender"], drop_first=True)

# 3. Target: keep readmitted as-is for now, we'll encode it right before training
df["readmitted"] = df["readmitted"].str.strip()

# 4. Drop columns we no longer need in raw form
df = df.drop(columns=["age"])

print("After encoding:", df.shape)
print(df.columns.tolist())

# --- Split ---
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,          # fixed seed = reproducible split every time you run this
    stratify=df["readmitted"] # preserves class proportions in both sets
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain class balance:")
print(train_df["readmitted"].value_counts(normalize=True))
print("\nTest class balance:")
print(test_df["readmitted"].value_counts(normalize=True))

In [7]:
import pandas as pd

# Read the local file directly — no azureml:// resolution needed, avoids the broken fsspec plugin
df = pd.read_parquet("./data_snapshot/diabetes_readmission_features.parquet")

print("Loaded:", df.shape)

Loaded: (101766, 7)


In [8]:
from sklearn.model_selection import train_test_split

# --- Clean/encode ---

# 1. Age: convert bracket strings like "[80-90)" into a numeric midpoint
def age_to_midpoint(age_bracket):
    low, high = age_bracket.strip("[)").split("-")
    return (int(low) + int(high)) / 2

df["age_numeric"] = df["age"].apply(age_to_midpoint)

# 2. Race/gender: one-hot encode
df = pd.get_dummies(df, columns=["race", "gender"], drop_first=True)

# 3. Target: clean whitespace
df["readmitted"] = df["readmitted"].str.strip()

# 4. Drop raw age column now that we have age_numeric
df = df.drop(columns=["age"])

print("After encoding:", df.shape)
print(df.columns.tolist())

# --- Split ---
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["readmitted"]
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain class balance:")
print(train_df["readmitted"].value_counts(normalize=True))
print("\nTest class balance:")
print(test_df["readmitted"].value_counts(normalize=True))

After encoding: (101766, 12)
['patient_nbr', 'time_in_hospital', 'num_medications', 'readmitted', 'age_numeric', 'race_AfricanAmerican', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other', 'gender_Male', 'gender_Unknown/Invalid']
Train shape: (81412, 12)
Test shape: (20354, 12)

Train class balance:
readmitted
NO     0.539122
>30    0.349285
<30    0.111593
Name: proportion, dtype: float64

Test class balance:
readmitted
NO     0.539108
>30    0.349268
<30    0.111624
Name: proportion, dtype: float64


In [9]:
import os
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

os.makedirs("./data_snapshot", exist_ok=True)

train_df.to_parquet("./data_snapshot/train.parquet", index=False)
test_df.to_parquet("./data_snapshot/test.parquet", index=False)

train_asset = Data(
    name="diabetes_readmission_train",
    version="1",
    description="Stratified 80% train split, encoded features",
    type=AssetTypes.URI_FILE,
    path="./data_snapshot/train.parquet"
)

test_asset = Data(
    name="diabetes_readmission_test",
    version="1",
    description="Stratified 20% test split, encoded features",
    type=AssetTypes.URI_FILE,
    path="./data_snapshot/test.parquet"
)

ml_client.data.create_or_update(train_asset)
ml_client.data.create_or_update(test_asset)

print("Train and test data assets registered.")

Uploading train.parquet (< 1 MB): 100%|██████████| 735k/735k [00:00<00:00, 12.2MB/s]


Uploading test.parquet (< 1 MB): 100%|██████████| 207k/207k [00:00<00:00, 11.3MB/s]




Train and test data assets registered.


training the basseline model 


In [10]:
import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

# Load the registered train/test splits (local files, same pattern as before)
train_df = pd.read_parquet("./data_snapshot/train.parquet")
test_df = pd.read_parquet("./data_snapshot/test.parquet")

# Separate features (X) from target (y)
target_col = "readmitted"
feature_cols = [c for c in train_df.columns if c not in [target_col, "patient_nbr"]]

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Features used:", feature_cols)
print("Train size:", X_train.shape, "Test size:", X_test.shape)

# Enable MLflow autologging for scikit-learn — this one line captures params, metrics, and the model itself
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="baseline_random_forest"):
    
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",   # up-weights minority classes automatically
        random_state=42,
        n_jobs=-1                  # use all available CPU cores on the compute instance
    )
    
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    
    # Print a full breakdown per class — more informative than a single accuracy number
    print(classification_report(y_test, preds))
    
    # Log a summary metric explicitly too (macro F1 = average F1 across all 3 classes, unweighted — good for imbalanced problems)
    macro_f1 = f1_score(y_test, preds, average="macro")
    mlflow.log_metric("macro_f1", macro_f1)
    print("Macro F1:", macro_f1)

print("Training run complete.")

Features used: ['time_in_hospital', 'num_medications', 'age_numeric', 'race_AfricanAmerican', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other', 'gender_Male', 'gender_Unknown/Invalid']
Train size: (81412, 10) Test size: (20354, 10)
              precision    recall  f1-score   support

         <30       0.13      0.42      0.20      2272
         >30       0.38      0.34      0.36      7109
          NO       0.61      0.39      0.47     10973

    accuracy                           0.37     20354
   macro avg       0.38      0.38      0.35     20354
weighted avg       0.48      0.37      0.40     20354

Macro F1: 0.3464753727405609
🏃 View run baseline_random_forest at: https://francecentral.api.azureml.ms/mlflow/v2.0/subscriptions/dcb792ae-5b43-4cfd-b612-b35b0dad827f/resourceGroups/rg-readmission-project/providers/Microsoft.MachineLearningServices/workspaces/aml-diabetes-fatima/#/experiments/70db5341-4484-4d41-ae35-28af2b936fb8/runs/d8208175-de0a-4b9f-9729-2ccda04c6d08
🧪

2026/07/22 06:17:09 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/07/22 06:17:13 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/anaconda/envs/azureml_py310_sdkv2

In [11]:
df = pd.read_sql("""
    SELECT p.patient_nbr, p.race, p.gender, p.age, 
           e.time_in_hospital, e.num_medications, e.num_lab_procedures, 
           e.num_procedures, e.number_outpatient, e.number_emergency, 
           e.number_inpatient, e.number_diagnoses,
           m.insulin, m.change, m.diabetesMed,
           r.readmitted 
    FROM dbo.patients p
    JOIN dbo.encounters e ON p.patient_nbr = e.patient_nbr
    JOIN dbo.medications m ON e.encounter_id = m.encounter_id
    JOIN dbo.readmission_labels r ON e.encounter_id = r.encounter_id
""", conn)

print(df.shape)
df.to_parquet("./data_snapshot/diabetes_readmission_features.parquet", index=False)

/tmp/ipykernel_3753/391135957.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


(101766, 16)


In [12]:
from sklearn.model_selection import train_test_split

# --- Clean/encode ---

# 1. Age: bracket string -> numeric midpoint
def age_to_midpoint(age_bracket):
    low, high = age_bracket.strip("[)").split("-")
    return (int(low) + int(high)) / 2

df["age_numeric"] = df["age"].apply(age_to_midpoint)

# 2. One-hot encode all categorical columns: race, gender, insulin, change, diabetesMed
df = pd.get_dummies(df, columns=["race", "gender", "insulin", "change", "diabetesMed"], drop_first=True)

# 3. Clean target
df["readmitted"] = df["readmitted"].str.strip()

# 4. Drop raw age
df = df.drop(columns=["age"])

print("After encoding:", df.shape)
print(df.columns.tolist())

# --- Split ---
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["readmitted"]
)

print("Train shape:", train_df.shape, "Test shape:", test_df.shape)

# Save locally (overwrite previous snapshot)
train_df.to_parquet("./data_snapshot/train.parquet", index=False)
test_df.to_parquet("./data_snapshot/test.parquet", index=False)
print("Saved new train/test splits.")

After encoding: (101766, 23)
['patient_nbr', 'time_in_hospital', 'num_medications', 'num_lab_procedures', 'num_procedures', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'readmitted', 'age_numeric', 'race_AfricanAmerican', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other', 'gender_Male', 'gender_Unknown/Invalid', 'insulin_No', 'insulin_Steady', 'insulin_Up', 'change_No', 'diabetesMed_Yes']
Train shape: (81412, 23) Test shape: (20354, 23)
Saved new train/test splits.


In [13]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

train_asset_v2 = Data(
    name="diabetes_readmission_train",
    version="2",
    description="Expanded features: added lab procedures, prior visits, insulin/med change flags",
    type=AssetTypes.URI_FILE,
    path="./data_snapshot/train.parquet"
)

test_asset_v2 = Data(
    name="diabetes_readmission_test",
    version="2",
    description="Expanded features: added lab procedures, prior visits, insulin/med change flags",
    type=AssetTypes.URI_FILE,
    path="./data_snapshot/test.parquet"
)

ml_client.data.create_or_update(train_asset_v2)
ml_client.data.create_or_update(test_asset_v2)
print("v2 data assets registered.")

Uploading train.parquet (< 1 MB): 100%|██████████| 1.03M/1.03M [00:00<00:00, 19.2MB/s]




v2 data assets registered.


In [14]:
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

train_df = pd.read_parquet("./data_snapshot/train.parquet")
test_df = pd.read_parquet("./data_snapshot/test.parquet")

target_col = "readmitted"
feature_cols = [c for c in train_df.columns if c not in [target_col, "patient_nbr"]]

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Features used:", feature_cols)

mlflow.sklearn.autolog()

with mlflow.start_run(run_name="random_forest_v2_expanded_features"):
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(classification_report(y_test, preds))
    macro_f1 = f1_score(y_test, preds, average="macro")
    mlflow.log_metric("macro_f1", macro_f1)
    print("Macro F1:", macro_f1)

Features used: ['time_in_hospital', 'num_medications', 'num_lab_procedures', 'num_procedures', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_numeric', 'race_AfricanAmerican', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other', 'gender_Male', 'gender_Unknown/Invalid', 'insulin_No', 'insulin_Steady', 'insulin_Up', 'change_No', 'diabetesMed_Yes']
              precision    recall  f1-score   support

         <30       0.19      0.35      0.25      2272
         >30       0.44      0.30      0.36      7109
          NO       0.64      0.67      0.66     10973

    accuracy                           0.51     20354
   macro avg       0.43      0.44      0.42     20354
weighted avg       0.52      0.51      0.51     20354

Macro F1: 0.4202332148911408
🏃 View run random_forest_v2_expanded_features at: https://francecentral.api.azureml.ms/mlflow/v2.0/subscriptions/dcb792ae-5b43-4cfd-b612-b35b0dad827f/resourceGroups/rg-readmission-project/provi

2026/07/22 06:27:43 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/07/22 06:27:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/anaconda/envs/azureml_py310_sdkv2

In [15]:
import os
os.makedirs("./train_component", exist_ok=True)

In [16]:
%%writefile ./train_component/train.py

import argparse
import os
import pandas as pd
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
import joblib

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_data", type=str, required=True, help="Path to train parquet file")
    parser.add_argument("--test_data", type=str, required=True, help="Path to test parquet file")
    parser.add_argument("--model_output", type=str, required=True, help="Directory to save the trained model")
    parser.add_argument("--n_estimators", type=int, default=200)
    parser.add_argument("--max_depth", type=int, default=10)
    args = parser.parse_args()

    mlflow.sklearn.autolog()

    train_df = pd.read_parquet(args.train_data)
    test_df = pd.read_parquet(args.test_data)

    target_col = "readmitted"
    feature_cols = [c for c in train_df.columns if c not in [target_col, "patient_nbr"]]

    X_train, y_train = train_df[feature_cols], train_df[target_col]
    X_test, y_test = test_df[feature_cols], test_df[target_col]

    with mlflow.start_run():
        model = RandomForestClassifier(
            n_estimators=args.n_estimators,
            max_depth=args.max_depth,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        print(classification_report(y_test, preds))

        macro_f1 = f1_score(y_test, preds, average="macro")
        mlflow.log_metric("macro_f1", macro_f1)
        print("Macro F1:", macro_f1)

        # Save model to the output directory Azure ML gives us
        os.makedirs(args.model_output, exist_ok=True)
        joblib.dump(model, os.path.join(args.model_output, "model.pkl"))
        print(f"Model saved to {args.model_output}")

if __name__ == "__main__":
    main()

Writing ./train_component/train.py


In [17]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

job = command(
    code="./train_component",          # folder containing train.py, gets uploaded
    command=(
        "python train.py "
        "--train_data ${{inputs.train_data}} "
        "--test_data ${{inputs.test_data}} "
        "--model_output ${{outputs.model_output}} "
        "--n_estimators 200 "
        "--max_depth 10"
    ),
    inputs={
        "train_data": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes_readmission_train:2"),
        "test_data": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes_readmission_test:2"),
    },
    outputs={
        "model_output": Output(type=AssetTypes.URI_FOLDER)
    },
    environment="azureml://registries/azureml/environments/sklearn-1.5/labels/latest",
    compute="cc-diabetes-training",   # your Compute Cluster from Phase 2
    display_name="rf_command_job_v1",
    experiment_name="diabetes_readmission_training"
)

returned_job = ml_client.jobs.create_or_update(job)
print("Job submitted. Job name:", returned_job.name)
print("Studio URL:", returned_job.studio_url)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading train_component (0.0 MBs): 1

Job submitted. Job name: brave_bucket_hshqgbq532
Studio URL: https://ml.azure.com/runs/brave_bucket_hshqgbq532?wsid=/subscriptions/dcb792ae-5b43-4cfd-b612-b35b0dad827f/resourcegroups/rg-readmission-project/workspaces/aml-diabetes-fatima&tid=e5aafe7c-971b-4ab7-b039-141ad36acec0


In [18]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model = Model(
    path=f"azureml://jobs/{returned_job.name}/outputs/model_output",
    name="diabetes_readmission_rf",
    description="RandomForest baseline, trained via Command Job, macro F1 ~0.42",
    type=AssetTypes.CUSTOM_MODEL,  # since we saved via joblib, not mlflow.sklearn.save_model
)

registered_model = ml_client.models.create_or_update(model)
print("Registered model name:", registered_model.name, "version:", registered_model.version)

Registered model name: diabetes_readmission_rf version: 1


In [1]:
import os
os.makedirs("./deploy_component", exist_ok=True)

In [2]:
%%writefile ./deploy_component/score.py

import os
import json
import joblib
import pandas as pd

def init():
    global model
    model_path = os.path.join(os.getenv("AZUREML_MODEL_DIR"), "model_output", "model.pkl")
    model = joblib.load(model_path)

def run(raw_data):
    try:
        data = json.loads(raw_data)
        df = pd.DataFrame(data["data"])
        preds = model.predict(df)
        return preds.tolist()
    except Exception as e:
        return {"error": str(e)}

Writing ./deploy_component/score.py


In [3]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import random

endpoint_name = "diabetes-readmit-endpoint"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Real-time readmission risk scoring",
    auth_mode="key"
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Endpoint '{endpoint_name}' created.")

NameError: name 'ml_client' is not defined

In [1]:
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential

credential = AzureCliCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id="dcb792ae-5b43-4cfd-b612-b35b0dad827f",
    resource_group_name="rg-readmission-project",
    workspace_name="aml-diabetes-fatima"
)
print("Connected.")

Connected.


In [2]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint_name = "diabetes-readmit-endpoint"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Real-time readmission risk scoring",
    auth_mode="key"
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Endpoint '{endpoint_name}' created.")

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Endpoint 'diabetes-readmit-endpoint' created.


In [3]:
from azure.ai.ml.entities import ManagedOnlineDeployment, Model, Environment, CodeConfiguration
from azure.ai.ml.constants import AssetTypes

# Reference your already-registered model
model = ml_client.models.get(name="diabetes_readmission_rf", version="1")

deployment = ManagedOnlineDeployment(
    name="blue",  # deployment slot name — "blue" is just a convention (blue/green deployments)
    endpoint_name=endpoint_name,
    model=model,
    code_configuration=CodeConfiguration(
        code="./deploy_component",
        scoring_script="score.py"
    ),
    environment="azureml://registries/azureml/environments/sklearn-1.5/labels/latest",
    instance_type="Standard_DS2_v2",   # small, cheap VM for serving
    instance_count=1
)

ml_client.online_deployments.begin_create_or_update(deployment).result()
print("Deployment 'blue' created.")

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint diabetes-readmit-endpoint exists
Uploading deploy_component (0.0 MBs): 100%|██████████| 443/443 [00:00<00:00, 46323.37it/s]




........................

HttpResponseError: (OutOfQuota) Not enough cluster CPU quota. The amount of additional CPU quota requested is 4, and you are currently using 4. Your amount of maximum quota is 6. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-outofquota
Code: OutOfQuota
Message: Not enough cluster CPU quota. The amount of additional CPU quota requested is 4, and you are currently using 4. Your amount of maximum quota is 6. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-outofquota

In [4]:
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=model,
    code_configuration=CodeConfiguration(
        code="./deploy_component",
        scoring_script="score.py"
    ),
    environment="azureml://registries/azureml/environments/sklearn-1.5/labels/latest",
    instance_type="Standard_DS1_v2",   # smaller: 1 vCPU instead of 2
    instance_count=1
)

ml_client.online_deployments.begin_create_or_update(deployment).result()
print("Deployment 'blue' created.")

Instance type Standard_DS1_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint diabetes-readmit-endpoint exists


HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"Specified deployment [blue] failed during initial provisioning and is in an unrecoverable state. Delete and re-create.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-6c0068d8f416320cec2b77362992efc1-739d0645b832b919-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"Specified deployment [blue] failed during initial provisioning and is in an unrecoverable state. Delete and re-create.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-6c0068d8f416320cec2b77362992efc1-739d0645b832b919-01\"}"}}
Additional I

In [5]:
ml_client.online_deployments.begin_delete(
    name="blue",
    endpoint_name=endpoint_name
).result()
print("Broken 'blue' deployment deleted.")

Broken 'blue' deployment deleted.


In [6]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

model = ml_client.models.get(name="diabetes_readmission_rf", version="1")

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=model,
    code_configuration=CodeConfiguration(
        code="./deploy_component",
        scoring_script="score.py"
    ),
    environment="azureml://registries/azureml/environments/sklearn-1.5/labels/latest",
    instance_type="Standard_DS1_v2",
    instance_count=1
)

ml_client.online_deployments.begin_create_or_update(deployment).result()
print("Deployment 'blue' created.")

Instance type Standard_DS1_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint diabetes-readmit-endpoint exists


.........................................................................................................................Deployment 'blue' created.


In [7]:
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Traffic routed to blue deployment.")

Traffic routed to blue deployment.


In [8]:
import json

sample_patient = {
    "data": [
        {
            "time_in_hospital": 5,
            "num_medications": 15,
            "num_lab_procedures": 40,
            "num_procedures": 1,
            "number_outpatient": 0,
            "number_emergency": 1,
            "number_inpatient": 2,
            "number_diagnoses": 7,
            "age_numeric": 65,
            "race_AfricanAmerican": False,
            "race_Asian": False,
            "race_Caucasian": True,
            "race_Hispanic": False,
            "race_Other": False,
            "gender_Male": True,
            "gender_Unknown/Invalid": False,
            "insulin_No": False,
            "insulin_Steady": True,
            "insulin_Up": False,
            "change_No": False,
            "diabetesMed_Yes": True
        }
    ]
}

request_file = "./sample_request.json"
with open(request_file, "w") as f:
    json.dump(sample_patient, f)

response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name="blue",
    request_file=request_file
)

print("Prediction:", response)

Prediction: ["<30"]


In [9]:
healthy_patient = {
    "data": [
        {
            "time_in_hospital": 1,
            "num_medications": 3,
            "num_lab_procedures": 10,
            "num_procedures": 0,
            "number_outpatient": 0,
            "number_emergency": 0,
            "number_inpatient": 0,
            "number_diagnoses": 2,
            "age_numeric": 35,
            "race_AfricanAmerican": False,
            "race_Asian": False,
            "race_Caucasian": True,
            "race_Hispanic": False,
            "race_Other": False,
            "gender_Male": False,
            "gender_Unknown/Invalid": False,
            "insulin_No": True,
            "insulin_Steady": False,
            "insulin_Up": False,
            "change_No": True,
            "diabetesMed_Yes": False
        }
    ]
}

with open(request_file, "w") as f:
    json.dump(healthy_patient, f)

response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name="blue",
    request_file=request_file
)

print("Prediction:", response)

Prediction: ["NO"]
